# RappiPlus | Data Quality Assessment

## 1. Project Overview

**Project:** RappiPlus — E-commerce Business Performance Analysis  
**Author:** Camila Leal Ledesma  
**Tools:** Python, Pandas, NumPy

### Business Context

RappiPlus is a subscription service within an e-commerce ecosystem designed to encourage repeat purchases and increase customer value.

The business needs to evaluate its profitability, understand customer purchasing behavior, identify conversion bottlenecks, and assess customer retention.

### Project Objective

Conduct an end-to-end analysis of business performance, using transactional, product, marketing, and behavioral data to identify opportunities for improvement.

### Purpose of This Notebook

Before calculating business metrics, assess the **quality and consistency of the available datasets** to ensure that subsequent analyses are based on reliable information.

### Data Sources

- `rappiplus_orders_raw.csv`: Customer orders and transactions.
- `rappiplus_catalog.csv`: Product information and unit costs.
- `rappiplus_marketing_spend.csv`: Marketing expenditures by country and channel.
- `experiment_checkout_ui.csv`: A/B testing data.

In [69]:
# Import libraries
import pandas as pd
import numpy as np

In [70]:
# Load raw datasets
orders = pd.read_csv('../data/raw/rappiplus_orders_raw.csv')
catalog = pd.read_csv('../data/raw/rappiplus_catalog.csv')
marketing = pd.read_csv('../data/raw/rappiplus_marketing_spend.csv')
experiment = pd.read_csv('../data/raw/experiment_checkout_ui.csv')

## 2. Initial Data Exploration

In this section, we examine the structure and content of each dataset before assessing data quality and identifying potential issues.

In [71]:
# Check dataset dimensions
print('Orders:', orders.shape)
print('Catalog:', catalog.shape)
print('Marketing:', marketing.shape)
print('Experiment:', experiment.shape)

Orders: (25100, 12)
Catalog: (7, 4)
Marketing: (1620, 5)
Experiment: (10000, 7)


In [72]:
# Preview orders dataset
orders.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [73]:
# Check column types and non-null values
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  str    
 1   id_usuario          25100 non-null  str    
 2   fecha_hora_pedido   25100 non-null  str    
 3   pais                24800 non-null  str    
 4   dispositivo         25080 non-null  str    
 5   fuente_referencia   25070 non-null  str    
 6   nombre_producto     25070 non-null  str    
 7   categoria_producto  25020 non-null  str    
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), str(8)
memory usage: 2.3 MB


In [74]:
# Check missing values
display(orders.isna().sum())
display(orders.isna().sum().sum())

id_pedido               0
id_usuario              0
fecha_hora_pedido       0
pais                  300
dispositivo            20
fuente_referencia      30
nombre_producto        30
categoria_producto     80
cantidad               50
precio_unitario        50
monto_descuento        50
monto_total             0
dtype: int64

np.int64(610)

In [75]:
# Check duplicate rows
orders.duplicated().sum()

np.int64(100)

In [76]:
# Check duplicate order IDs
orders['id_pedido'].duplicated().sum()

np.int64(100)

In [77]:
# Check descriptive statistics
orders.describe()

,cantidad,precio_unitario,monto_descuento,monto_total
count,25050.000000,25050.000000,25050.000000,2.510000e+04
mean,7.092735,259.305549,4.500798,2.072680e+03
std,296.277003,138.726461,5.223010,9.894995e+04
min,-2.000000,20.030000,0.000000,-4.926500e+02
25%,1.000000,138.377500,0.000000,1.805075e+02
50%,2.000000,258.715000,0.000000,3.417500e+02
75%,2.000000,380.332500,10.000000,5.185800e+02
max,20000.000000,499.960000,15.000000,8.840200e+06


In [78]:
# Inspect orders with unusual quantities
orders[(orders['cantidad'] < 0) | (orders['cantidad'] > 100)]

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
266,order_266,user_7011,2025-03-13,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,-2.0,101.31,10.0,-192.62
267,order_267,user_1087,2025-05-07,NaN,desktop,social,Phone-Pro-128GB,Electronica,-1.0,43.50,5.0,-38.50
268,order_268,user_84,2025-02-19,NaN,desktop,organic,Phone-Pro-128GB,Electronica,-1.0,497.65,5.0,-492.65
269,order_269,user_3323,2025-05-25,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,-1.0,423.53,0.0,-423.53
3521,order_3521,user_5812,2025-02-03,Mexico,mobile,paid_search,Laptop-Gaming-16GB,Electronica,10000.0,43.14,0.0,431400.00
3522,order_3522,user_3575,2025-03-29,Argentina,desktop,social,Laptop-Gaming-16GB,Electronica,10000.0,280.55,0.0,2805500.00
3586,order_3586,user_3380,2025-02-03,Mexico,mobile,paid_search,Laptop-Gaming-16GB,Electronica,10000.0,490.35,0.0,4903500.00
3643,order_3643,user_4440,2025-01-07,Colombia,desktop,social,Laptop-Gaming-16GB,Electronica,10000.0,238.15,0.0,2381500.00
3656,order_3656,user_884,2025-01-01,Argentina,mobile,organic,Laptop-Gaming-16GB,Electronica,20000.0,297.66,0.0,5953200.00
3668,order_3668,user_7270,2025-06-24,Mexico,mobile,paid_search,Laptop-Gaming-16GB,Electronica,20000.0,348.31,0.0,6966200.00


In [79]:
# Check order amount consistency
orders['monto_esperado'] = (
    orders['cantidad'] * orders['precio_unitario']
    - orders['monto_descuento']
)

orders['diferencia_monto'] = (
    orders['monto_total'] - orders['monto_esperado']
)

orders[
    orders['diferencia_monto'].abs() > 0.01
].head(10)

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,monto_esperado,diferencia_monto
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99,195.98,0.01
24,order_24,user_458,2025-01-05,Colombia,NaN,organic,Sneakers-Urban-42,Moda,2.0,117.99,0.0,235.99,235.98,0.01
35,order_35,user_3817,2025-02-08,Mexico,NaN,organic,Blender-XL-Red,Hogar,2.0,467.34,5.0,929.69,929.68,0.01
41,order_41,user_2963,2025-01-27,Mexico,NaN,social,Jacket-Winter-M,Moda,2.0,37.77,10.0,65.53,65.54,-0.01
136,order_136,user_6937,2025-02-10,NaN,desktop,organic,Blender-XL-Red,Hogar,2.0,211.05,15.0,407.09,407.10,-0.01
180,order_180,user_203,2025-05-29,NaN,desktop,paid_search,Sneakers-Urban-42,Moda,2.0,451.16,15.0,887.31,887.32,-0.01
261,order_261,user_5138,2025-05-07,NaN,desktop,organic,Phone-Pro-128GB,Electronica,2.0,209.33,10.0,408.65,408.66,-0.01
266,order_266,user_7011,2025-03-13,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,-2.0,101.31,10.0,-192.62,-212.62,20.00
267,order_267,user_1087,2025-05-07,NaN,desktop,social,Phone-Pro-128GB,Electronica,-1.0,43.50,5.0,-38.50,-48.50,10.00
268,order_268,user_84,2025-02-19,NaN,desktop,organic,Phone-Pro-128GB,Electronica,-1.0,497.65,5.0,-492.65,-502.65,10.00


In [80]:
# Count orders with significant amount differences
(orders['diferencia_monto'].abs() > 0.02).sum()

np.int64(3)

In [81]:
# Inspect orders with significant amount differences
orders[orders['diferencia_monto'].abs() > 0.02]

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,monto_esperado,diferencia_monto
266,order_266,user_7011,2025-03-13,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,-2.0,101.31,10.0,-192.62,-212.62,20.0
267,order_267,user_1087,2025-05-07,NaN,desktop,social,Phone-Pro-128GB,Electronica,-1.0,43.50,5.0,-38.50,-48.50,10.0
268,order_268,user_84,2025-02-19,NaN,desktop,organic,Phone-Pro-128GB,Electronica,-1.0,497.65,5.0,-492.65,-502.65,10.0


In [82]:
# Calculate expected amounts considering negative quantities
orders['monto_esperado_ajustado'] = np.where(
    orders['cantidad'] < 0,
    orders['cantidad'] * orders['precio_unitario'] + orders['monto_descuento'],
    orders['cantidad'] * orders['precio_unitario'] - orders['monto_descuento']
)

# Check remaining amount differences
orders['diferencia_ajustada'] = (
    orders['monto_total'] - orders['monto_esperado_ajustado']
)

(orders['diferencia_ajustada'].abs() > 0.02).sum()

np.int64(0)

In [83]:
# Count orders that could not be validated
orders['monto_esperado_ajustado'].isna().sum()

np.int64(50)

In [84]:
# Inspect orders with incomplete amount information
orders[
    orders['monto_esperado_ajustado'].isna()
].head(10)

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,monto_esperado,diferencia_monto,monto_esperado_ajustado,diferencia_ajustada
74,order_74,user_6172,2025-01-15,Argentina,desktop,organic,Sneakers-Urban-42,NaN,NaN,NaN,NaN,595.85,NaN,NaN,NaN,NaN
75,order_75,user_6588,2025-06-19,Colombia,mobile,organic,Sneakers-Urban-42,NaN,NaN,NaN,NaN,458.15,NaN,NaN,NaN,NaN
76,order_76,user_3193,2025-03-17,Argentina,mobile,paid_search,Laptop-Gaming-16GB,NaN,NaN,NaN,NaN,319.75,NaN,NaN,NaN,NaN
77,order_77,user_775,2025-06-24,Argentina,desktop,social,Vacuum-Pro-Black,NaN,NaN,NaN,NaN,227.55,NaN,NaN,NaN,NaN
78,order_78,user_2702,2025-03-19,Argentina,desktop,paid_search,Phone-Pro-128GB,NaN,NaN,NaN,NaN,432.39,NaN,NaN,NaN,NaN
79,order_79,user_6438,2025-03-22,Colombia,desktop,paid_search,Sneakers-Urban-42,NaN,NaN,NaN,NaN,527.15,NaN,NaN,NaN,NaN
80,order_80,user_6790,2025-01-03,Mexico,desktop,organic,Vacuum-Pro-Black,NaN,NaN,NaN,NaN,711.18,NaN,NaN,NaN,NaN
81,order_81,user_232,2025-05-16,Mexico,mobile,paid_search,Tablet-Standard-64GB,NaN,NaN,NaN,NaN,41.08,NaN,NaN,NaN,NaN
82,order_82,user_5665,2025-02-07,Colombia,mobile,paid_search,Sneakers-Urban-42,NaN,NaN,NaN,NaN,431.55,NaN,NaN,NaN,NaN
83,order_83,user_5225,2025-04-25,Argentina,mobile,organic,Sneakers-Urban-42,NaN,NaN,NaN,NaN,279.35,NaN,NaN,NaN,NaN


In [85]:
# Check missing values in orders with incomplete amounts
orders[
    orders['monto_esperado_ajustado'].isna()
][
    ['cantidad', 'precio_unitario', 'monto_descuento',
     'categoria_producto', 'monto_total']
].isna().sum()

cantidad              50
precio_unitario       50
monto_descuento       50
categoria_producto    50
monto_total            0
dtype: int64

In [86]:
# Check products in orders with incomplete amounts
orders[
    orders['monto_esperado_ajustado'].isna()
]['nombre_producto'].value_counts(dropna=False)

nombre_producto
Sneakers-Urban-42       19
Jacket-Winter-M         10
Vacuum-Pro-Black         6
Tablet-Standard-64GB     5
Laptop-Gaming-16GB       4
Phone-Pro-128GB          3
Blender-XL-Red           3
Name: count, dtype: int64

In [87]:
# Check unique values in categorical columns
orders['pais'].value_counts(dropna=False)

pais
Colombia     7520
Mexico       7502
Argentina    7291
mexico        865
colombia      823
argentina     799
NaN           300
Name: count, dtype: int64

In [88]:
# Check categorical values
categorical_columns = [
    'dispositivo',
    'fuente_referencia',
    'nombre_producto',
    'categoria_producto'
]

for column in categorical_columns:
    print(f'\n{column}')
    print(orders[column].value_counts(dropna=False))


dispositivo
dispositivo
desktop    12759
mobile     12321
NaN           20
Name: count, dtype: int64

fuente_referencia
fuente_referencia
social         8428
organic        8329
paid_search    8313
NaN              30
Name: count, dtype: int64

nombre_producto
nombre_producto
Vacuum-Pro-Black        4199
Blender-XL-Red          4195
Jacket-Winter-M         4192
Sneakers-Urban-42       4160
Laptop-Gaming-16GB      2794
Tablet-Standard-64GB    2780
Phone-Pro-128GB         2750
NaN                       30
Name: count, dtype: int64

categoria_producto
categoria_producto
Hogar          8385
Moda           8323
Electronica    8312
NaN              80
Name: count, dtype: int64


In [89]:
# Check order dates
orders['fecha_hora_pedido'].head()

0    2025-05-22
1    2025-06-15
2    2025-05-02
3    2025-06-09
4    2025-03-30
Name: fecha_hora_pedido, dtype: str

In [90]:
# Check invalid dates
pd.to_datetime(
    orders['fecha_hora_pedido'],
    errors='coerce'
).isna().sum()

np.int64(0)

In [91]:
# Check order date range
dates = pd.to_datetime(orders['fecha_hora_pedido'])

print('Earliest date:', dates.min())
print('Latest date:', dates.max())

Earliest date: 2025-01-01 00:00:00
Latest date: 2025-06-30 00:00:00


In [92]:
# Count orders with unusual quantities
print('Negative quantities:', (orders['cantidad'] < 0).sum())
print('Zero quantities:', (orders['cantidad'] == 0).sum())
print('Quantities above 100:', (orders['cantidad'] > 100).sum())

Negative quantities: 4
Zero quantities: 0
Quantities above 100: 10


In [93]:
# Check revenue from orders with unusually high quantities
high_quantity_orders = orders[orders['cantidad'] > 100]

print('Total recorded revenue:', orders['monto_total'].sum())
print('Revenue from high-quantity orders:', high_quantity_orders['monto_total'].sum())

Total recorded revenue: 52024257.55999999
Revenue from high-quantity orders: 42344700.0


In [94]:
# Compare recorded amounts with and without high-quantity orders

total_amount = orders['monto_total'].sum()

amount_without_outliers = orders.loc[
    orders['cantidad'] <= 100, 'monto_total'
].sum()

print('Total recorded amount:', round(total_amount, 2))
print('Amount without high-quantity orders:', round(amount_without_outliers, 2))

Total recorded amount: 52024257.56
Amount without high-quantity orders: 9657929.56


In [95]:
# Compare amounts excluding only high-quantity orders

amount_without_high_quantities = orders.loc[
    (orders['cantidad'] <= 100) | (orders['cantidad'].isna()),
    'monto_total'
].sum()

print('Total recorded amount:', round(total_amount, 2))
print('Amount excluding high-quantity orders:', round(amount_without_high_quantities, 2))

Total recorded amount: 52024257.56
Amount excluding high-quantity orders: 9679557.56


### Preliminary Findings: Order Data Quality

The initial assessment identified:

- **100 duplicate rows**, which require removal after validation.
- **610 missing values** across eight columns, including 50 orders with incomplete transaction details.
- **4 orders with negative quantities**, potentially representing returns.
- **10 orders with unusually high quantities**, accounting for approximately 81.4% of the total recorded amount.
- **Inconsistent country names** due to capitalization differences.
- **Valid order dates** within the expected period (January–June 2025).

The high-quantity orders have a significant impact on the recorded amount. Their validity must be assessed before calculating final business metrics.

No records have been removed or corrected at this stage.

## 3. Product Catalog Assessment

This section examines the product catalog to assess data completeness, consistency, and product uniqueness before using it in the business analysis.

In [96]:
# Preview product catalog
catalog

,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"
5,Sneakers-Urban-42,Moda,17.21,Greene-Smith
6,Jacket-Winter-M,Moda,189.31,Mcmillan-Rhodes


In [97]:
# Check catalog structure and data types
catalog.info()

<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      str    
 1   categoria_producto  7 non-null      str    
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      str    
dtypes: float64(1), str(3)
memory usage: 356.0 bytes


In [98]:
# Check product uniqueness and duplicate rows
print('Duplicate products:', catalog['nombre_producto'].duplicated().sum())
print('Duplicate rows:', catalog.duplicated().sum())

Duplicate products: 0
Duplicate rows: 0


In [99]:
# Check invalid unit costs
catalog[catalog['costo_unitario'] <= 0]

,nombre_producto,categoria_producto,costo_unitario,proveedor


In [100]:
# Check products missing from the catalog
orders.loc[
    ~orders['nombre_producto'].isin(catalog['nombre_producto'])
    & orders['nombre_producto'].notna(),
    'nombre_producto'
].unique()

<StringArray>
[]
Length: 0, dtype: str

In [101]:
# Compare product categories between orders and catalog
category_comparison = orders[
    ['nombre_producto', 'categoria_producto']
].drop_duplicates().merge(
    catalog[['nombre_producto', 'categoria_producto']],
    on='nombre_producto',
    how='left',
    suffixes=('_orders', '_catalog')
)

category_comparison

,nombre_producto,categoria_producto_orders,categoria_producto_catalog
0,Jacket-Winter-M,Moda,Moda
1,Tablet-Standard-64GB,Electronica,Electrónica
2,Blender-XL-Red,Hogar,Hogar
3,Laptop-Gaming-16GB,Electronica,Electrónica
4,Sneakers-Urban-42,Moda,Moda
5,Phone-Pro-128GB,Electronica,Electrónica
6,Vacuum-Pro-Black,Hogar,Hogar
7,NaN,NaN,NaN
8,Sneakers-Urban-42,NaN,Moda
9,Laptop-Gaming-16GB,NaN,Electrónica


In [102]:
# Count recoverable product categories
orders[
    orders['categoria_producto'].isna()
    & orders['nombre_producto'].notna()
].shape[0]

50

### Preliminary Findings: Product Catalog

The catalog contains 7 unique products, with no missing values, duplicate records, or invalid unit costs.

All identified products in the orders dataset have a matching entry in the catalog.

The comparison revealed inconsistent spelling of the Electronics category and 80 orders with missing product categories. Of these, 50 categories can be recovered using the catalog, while 30 require further investigation.

The catalog provides a reliable reference for product categories and unit costs, subject to the remaining cross-table validations.

## 4. Marketing Data Assessment

This section examines marketing spending data to identify potential quality issues before evaluating marketing performance and profitability.

In [103]:
# Preview marketing dataset
marketing.head()

,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


In [104]:
# Check marketing structure and data types
marketing.info()

<class 'pandas.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   str    
 1   pais        1620 non-null   str    
 2   id_campaña  1620 non-null   str    
 3   canal       1519 non-null   str    
 4   gasto       1620 non-null   float64
dtypes: float64(1), str(4)
memory usage: 63.4 KB


In [105]:
# Check duplicate marketing records
print('Duplicate rows:', marketing.duplicated().sum())

print(
    'Duplicate date-campaign combinations:',
    marketing.duplicated(subset=['fecha', 'id_campaña']).sum()
)

Duplicate rows: 0
Duplicate date-campaign combinations: 0


In [106]:
# Inspect marketing records with missing channels
marketing[marketing['canal'].isna()].head(10)

,fecha,pais,id_campaña,canal,gasto
98,2025-01-11,Argentina,social_Argentina,NaN,849.70
99,2025-01-12,Mexico,organic_Mexico,NaN,2033.56
100,2025-01-12,Mexico,paid_search_Mexico,NaN,1260.65
101,2025-01-12,Mexico,social_Mexico,NaN,1660.90
102,2025-01-12,Colombia,organic_Colombia,NaN,1819.27
103,2025-01-12,Colombia,paid_search_Colombia,NaN,1583.33
104,2025-01-12,Colombia,social_Colombia,NaN,712.64
105,2025-01-12,Argentina,organic_Argentina,NaN,1860.40
106,2025-01-12,Argentina,paid_search_Argentina,NaN,885.06
107,2025-01-12,Argentina,social_Argentina,NaN,2253.77


In [107]:
# Check channel consistency by campaign
marketing.groupby('id_campaña')['canal'].nunique()

id_campaña
organic_Argentina        1
organic_Colombia         1
organic_Mexico           1
paid_search_Argentina    1
paid_search_Colombia     1
paid_search_Mexico       1
social_Argentina         1
social_Colombia          1
social_Mexico            1
Name: canal, dtype: int64

In [108]:
# Check recoverable missing channels
campaign_channels = marketing.groupby('id_campaña')['canal'].first()

missing_channels = marketing[marketing['canal'].isna()]

missing_channels['id_campaña'].isin(campaign_channels.index).sum()

np.int64(101)

In [109]:
# Check marketing dates
marketing_dates = pd.to_datetime(marketing['fecha'], errors='coerce')

print('Invalid dates:', marketing_dates.isna().sum())
print('Earliest date:', marketing_dates.min())
print('Latest date:', marketing_dates.max())

Invalid dates: 0
Earliest date: 2025-01-01 00:00:00
Latest date: 2025-06-29 00:00:00


In [110]:
# Check marketing spending
marketing['gasto'].describe()

count    1620.00000
mean     1772.74292
std       734.43294
min       501.11000
25%      1128.03000
50%      1782.42500
75%      2420.68500
max      2999.36000
Name: gasto, dtype: float64

In [111]:
# Check marketing countries and channels
print('Countries:')
print(marketing['pais'].value_counts(dropna=False))

print('\nChannels:')
print(marketing['canal'].value_counts(dropna=False))

Countries:
pais
Mexico       540
Colombia     540
Argentina    540
Name: count, dtype: int64

Channels:
canal
paid_search    507
organic        506
social         506
NaN            101
Name: count, dtype: int64


### Preliminary Findings: Marketing Data

The marketing dataset contains 1,620 records covering Argentina, Colombia, and Mexico.

- No duplicate rows or duplicate date-campaign combinations were found.
- All dates are valid and fall between January 1 and June 29, 2025.
- Marketing spending values are positive, with no obvious extreme values.
- Country and channel names are consistent.
- There are 101 missing channel values, all of which can be recovered using the existing campaign-to-channel mapping.

The dataset is suitable for further analysis after recovering missing channels and converting dates to the appropriate format.

Marketing data ends one day before the orders dataset, which should be considered when comparing business performance across both sources.

No records have been modified at this stage.

## 5. A/B Experiment Data Assessment

This section examines the experiment dataset to assess data quality and consistency before analyzing conversion results.

In [112]:
# Preview experiment dataset
experiment.head()

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [113]:
# Check experiment structure and data types
experiment.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_usuario       10000 non-null  str    
 1   variante         10000 non-null  str    
 2   convirtio        10000 non-null  int64  
 3   dispositivo      10000 non-null  str    
 4   pais             10000 non-null  str    
 5   duracion_sesion  10000 non-null  float64
 6   timestamp        10000 non-null  str    
dtypes: float64(1), int64(1), str(5)
memory usage: 547.0 KB


In [114]:
# Check duplicate experiment records
print('Duplicate rows:', experiment.duplicated().sum())

print('Duplicate user IDs:', experiment['id_usuario'].duplicated().sum())

Duplicate rows: 0
Duplicate user IDs: 0


In [115]:
# Check experiment variants and conversion values
print('Experiment variants:')
print(experiment['variante'].value_counts(dropna=False))

print('\nConversion values:')
print(experiment['convirtio'].value_counts(dropna=False))

Experiment variants:
variante
tratamiento    5035
control        4965
Name: count, dtype: int64

Conversion values:
convirtio
0    8401
1    1599
Name: count, dtype: int64


In [116]:
# Check experiment dates
experiment_dates = pd.to_datetime(
    experiment['timestamp'],
    errors='coerce'
)

print('Invalid dates:', experiment_dates.isna().sum())
print('Earliest date:', experiment_dates.min())
print('Latest date:', experiment_dates.max())

Invalid dates: 0
Earliest date: 2025-01-01 00:00:00
Latest date: 2025-06-30 00:00:00


In [117]:
# Check session duration statistics
experiment['duracion_sesion'].describe()

count    10000.000000
mean       159.862439
std         81.074410
min         20.010000
25%         89.470000
50%        159.725000
75%        229.745000
max        300.000000
Name: duracion_sesion, dtype: float64

In [118]:
# Check experiment countries and devices
print('Countries:')
print(experiment['pais'].value_counts(dropna=False))

print('\nDevices:')
print(experiment['dispositivo'].value_counts(dropna=False))

Countries:
pais
Mexico       3405
Argentina    3317
Colombia     3278
Name: count, dtype: int64

Devices:
dispositivo
desktop    5042
mobile     4958
Name: count, dtype: int64


### Preliminary Findings: A/B Experiment

The experiment dataset contains 10,000 unique users distributed across control (4,965) and treatment (5,035) groups.

- No missing values or duplicate records were identified.
- Each user appears only once in the dataset.
- The conversion variable contains only valid binary values (0 and 1).
- All dates are valid and fall between January 1 and June 30, 2025.
- Session durations are positive, with no obvious extreme values.
- Country and device categories are consistent.

The dataset shows no major quality issues in the checks performed.

Before evaluating the experiment's effectiveness, group comparability and the experimental design should be assessed.

## 6. Cross-Table Validation

This section validates the relationships between datasets before data cleaning and integration.

In [119]:
# Validate the relationship between orders and catalog

orders_catalog = orders.merge(
    catalog[['nombre_producto', 'costo_unitario']],
    on='nombre_producto',
    how='left',
    validate='many_to_one'
)

print('Original orders:', len(orders))
print('Orders after merge:', len(orders_catalog))
print('Missing unit costs:', orders_catalog['costo_unitario'].isna().sum())

Original orders: 25100
Orders after merge: 25100
Missing unit costs: 30


In [120]:
# Validate product category recovery

category_validation = orders.merge(
    catalog[['nombre_producto', 'categoria_producto']],
    on='nombre_producto',
    how='left',
    suffixes=('_orders', '_catalog'),
    validate='many_to_one'
)

print(
    'Recoverable missing categories:',
    (
        category_validation['categoria_producto_orders'].isna()
        & category_validation['categoria_producto_catalog'].notna()
    ).sum()
)

print(
    'Categories still missing:',
    category_validation['categoria_producto_catalog'].isna().sum()
)

Recoverable missing categories: 50
Categories still missing: 30


### Orders and Catalog: Cross-Table Validation

The relationship between orders and the product catalog was validated using a many-to-one merge.

* All 25,100 order records were preserved without introducing additional rows.
* All orders with identified products matched the catalog.
* Product categories can be recovered for 50 of the 80 orders with missing categories.
* The remaining 30 orders have no product identifier and cannot be matched directly to the catalog.
* Unit costs are available for orders with identified products, although missing quantities prevent cost calculation for some transactions.

These findings support using the product catalog as a reference for recovering missing categories and calculating product costs during data preparation.

No changes have been made to the original datasets.


In [121]:
# Compare countries and channels between datasets

print('Countries in orders:')
print(orders['pais'].unique())

print('\nCountries in marketing:')
print(marketing['pais'].unique())

print('\nChannels in orders:')
print(orders['fuente_referencia'].unique())

print('\nChannels in marketing:')
print(marketing['canal'].unique())

Countries in orders:
<StringArray>
['Argentina', 'Mexico', 'Colombia', 'mexico', 'colombia', 'argentina', nan]
Length: 7, dtype: str

Countries in marketing:
<StringArray>
['Mexico', 'Colombia', 'Argentina']
Length: 3, dtype: str

Channels in orders:
<StringArray>
['organic', 'paid_search', 'social', nan]
Length: 4, dtype: str

Channels in marketing:
<StringArray>
['organic', 'paid_search', 'social', nan]
Length: 4, dtype: str


In [122]:
# Check marketing records by country and channel
marketing.groupby(
    ['pais', 'canal'],
    dropna=False
).size()

pais       canal      
Argentina  organic        169
           paid_search    169
           social         168
           NaN             34
Colombia   organic        169
           paid_search    169
           social         169
           NaN             33
Mexico     organic        168
           paid_search    169
           social         169
           NaN             34
dtype: int64

In [123]:
# Prepare temporary copies for cross-table validation
orders_check = orders.copy()
marketing_check = marketing.copy()

# Standardize country names
orders_check['pais'] = orders_check['pais'].str.capitalize()

# Convert dates
orders_check['fecha'] = pd.to_datetime(
    orders_check['fecha_hora_pedido']
)

marketing_check['fecha'] = pd.to_datetime(
    marketing_check['fecha']
)

# Recover marketing channels using campaign information
marketing_check['canal'] = marketing_check['canal'].fillna(
    marketing_check['id_campaña'].map(campaign_channels)
)

In [124]:
# Check matching date-country-channel combinations

orders_keys = orders_check[
    ['fecha', 'pais', 'fuente_referencia']
].dropna().drop_duplicates()

marketing_keys = marketing_check[
    ['fecha', 'pais', 'canal']
].dropna().drop_duplicates()

matching_keys = orders_keys.merge(
    marketing_keys,
    left_on=['fecha', 'pais', 'fuente_referencia'],
    right_on=['fecha', 'pais', 'canal'],
    how='left',
    indicator=True
)

print('Order combinations:', len(orders_keys))
print('Matching combinations:', (matching_keys['_merge'] == 'both').sum())
print('Unmatched combinations:', (matching_keys['_merge'] == 'left_only').sum())

Order combinations: 1629
Matching combinations: 1620
Unmatched combinations: 9


In [125]:
# Inspect unmatched date-country-channel combinations

matching_keys.loc[
    matching_keys['_merge'] == 'left_only',
    ['fecha', 'pais', 'fuente_referencia']
]

,fecha,pais,fuente_referencia
245,2025-06-30,Mexico,paid_search
410,2025-06-30,Argentina,organic
564,2025-06-30,Argentina,paid_search
622,2025-06-30,Colombia,paid_search
953,2025-06-30,Argentina,social
1079,2025-06-30,Mexico,social
1080,2025-06-30,Mexico,organic
1133,2025-06-30,Colombia,organic
1336,2025-06-30,Colombia,social


In [126]:
# Count orders without matching marketing records

unmatched_keys = matching_keys.loc[
    matching_keys['_merge'] == 'left_only',
    ['fecha', 'pais', 'fuente_referencia']
]

unmatched_orders = orders_check.merge(
    unmatched_keys,
    on=['fecha', 'pais', 'fuente_referencia'],
    how='inner'
)

print('Orders without matching marketing data:', len(unmatched_orders))
print('Recorded amount:', round(unmatched_orders['monto_total'].sum(), 2))

Orders without matching marketing data: 126
Recorded amount: 47365.04


### Orders and Marketing: Cross-Table Validation

The relationship between orders and marketing spending was assessed using date, country, and acquisition channel.

- Country and channel values are compatible after standardization.
- Marketing channel information can be recovered for 101 records using campaign identifiers.
- Of 1,629 unique date-country-channel combinations in orders, 1,620 matched the marketing dataset.
- The 9 unmatched combinations correspond to June 30, 2025, which is not covered by the marketing dataset.
- These combinations represent 126 order records and a recorded amount of 47,365.04.

Orders from June 30 can be retained for sales analysis, but marketing efficiency metrics requiring spending data should be calculated only for periods with comparable data coverage.

Records with missing country or acquisition channel information require separate consideration.

No original datasets were modified.

In [127]:
# Check shared user IDs between orders and experiment

shared_users = set(orders['id_usuario'].dropna()) & set(
    experiment['id_usuario'].dropna()
)

print('Unique users in orders:', orders['id_usuario'].nunique())
print('Unique users in experiment:', experiment['id_usuario'].nunique())
print('Shared users:', len(shared_users))

Unique users in orders: 7642
Unique users in experiment: 10000
Shared users: 0


### Orders and Experiment: Cross-Table Validation

The orders dataset contains 7,642 unique users, while the A/B experiment contains 10,000 unique users.

No matching user IDs were found between the two datasets.

Therefore, the experiment cannot be directly linked to recorded orders using the available identifiers.

The A/B test will be analyzed independently using its conversion variable, while orders, catalog, and marketing data will support the business performance analysis.

A mapping between user identifiers would be required to establish a direct relationship between the experiment and recorded purchases.